# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.8 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:

import os, json, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

try:
    torch.set_num_threads(1)
except Exception:
    pass

TASK_ID = 'task370'
CH, H, W = 10, 30, 30
COMPETITION = Path('/kaggle/input/competitions/neurogolf-2026')
WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT_DIR = WORKDIR / 'task370_qstep_relation_bank_v6_onnx'
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_JSON = Path('/mnt/data/task370.json')
TASK_JSON = LOCAL_JSON if LOCAL_JSON.exists() else (COMPETITION / f'{TASK_ID}.json')
assert TASK_JSON.exists(), f'Missing task file: {TASK_JSON}'

ONNX_PATH = OUT_DIR / f'{TASK_ID}.onnx'

KAGGLE_SUBMISSION_ZIP = WORKDIR / 'submission.zip'
BUNDLE_ZIP = WORKDIR / 'arc_task_notebook_task370_v6_qstep_relation_bank.zip'

with open(TASK_JSON) as f:
    task = json.load(f)
print('Loaded', TASK_JSON)
print('train/test/arc-gen:', len(task.get('train', [])), len(task.get('test', [])), len(task.get('arc-gen', [])))


Loaded /kaggle/input/competitions/neurogolf-2026/task370.json
train/test/arc-gen: 3 1 262


In [6]:

def grid_to_tensor(grid, ch=CH, h=H, w=W):
    arr = np.asarray(grid, dtype=np.int64)
    assert arr.ndim == 2
    t = np.zeros((1, ch, h, w), dtype=np.float32)
    gh, gw = arr.shape
    assert gh <= h and gw <= w
    for r in range(gh):
        for c in range(gw):
            v = int(arr[r, c])
            assert 0 <= v < ch
            t[0, v, r, c] = 1.0
    return t

def tensor_to_grid_argmax(t):
    return np.asarray(t).argmax(axis=1)[0]

def active_canvas_from_tensor(t):
    return (t.sum(axis=1, keepdims=True) > 0).astype(np.float32)


In [7]:

class Task370QStepRelationBank(nn.Module):
    """Nonlocal q/step relation-bank solver for task370.

    Structural interpretation:
    - active canvas contains one dominant nonzero background color, a zero-template, and one seed pixel;
    - output preserves all original pixels;
    - added pixels copy the zero-template mask in the seed color;
    - continuation is controlled by direction d, reference zero-cell q, and step size s.

    This model keeps q/s as explicit regimes:
    - corner marker: q is the opposite template corner; step s is template bbox size;
    - side marker: q is top/bottom middle; initial offset is 2*d, then sparse templates use step 1 and plus templates use step 2.

    The graph is static: all shifts are pre-unrolled and masked, no Loop/Scan/NonZero/Unique.
    """
    def __init__(self):
        super().__init__()
        dirs = [(1,1), (1,-1), (-1,1), (-1,-1)]
        shifts = set()

        # Corner block tiling: marker sits one block away from the reference corner q.
        for sr, sc in dirs:
            for s in [3, 4, 5]:
                vr, vc = sr * s, sc * s
                shifts.add((-vr, -vc))  # marker gate shift
                for k in range(1, 31):
                    dr, dc = k * vr, k * vc
                    if abs(dr) >= 30 or abs(dc) >= 30:
                        break
                    shifts.add((dr, dc))

        # Side overlap: marker is q + 2*d, then copy spacing is 1 or 2.
        for sr, sc in dirs:
            ir, ic = 2 * sr, 2 * sc
            shifts.add((-ir, -ic))
            for step in [1, 2]:
                for n in range(0, 31):
                    dr, dc = ir + n * step * sr, ic + n * step * sc
                    if abs(dr) >= 30 or abs(dc) >= 30:
                        break
                    shifts.add((dr, dc))

        for dr, dc in sorted(shifts):
            mask = torch.zeros(1, 1, 30, 30, dtype=torch.float32)
            r0, r1 = max(0, dr), 30 + min(0, dr)
            c0, c1 = max(0, dc), 30 + min(0, dc)
            if r1 > r0 and c1 > c0:
                mask[:, :, r0:r1, c0:c1] = 1.0
            self.register_buffer(f'mask_{dr+40}_{dc+40}', mask)

        nonzero_color = torch.ones(1, 10, 1, 1, dtype=torch.float32)
        nonzero_color[:, 0:1] = 0.0
        self.register_buffer('nonzero_color', nonzero_color)

    def shift(self, z, dr: int, dc: int):
        return torch.roll(z, shifts=(dr, dc), dims=(2, 3)) * getattr(self, f'mask_{dr+40}_{dc+40}')

    def endpoint_masks(self, zero):
        # Nonlocal top/bottom rows and left/right columns of the zero-template bbox.
        row_has = (zero.sum(dim=3, keepdim=True) > 0.5).float()
        col_has = (zero.sum(dim=2, keepdim=True) > 0.5).float()

        top_row = row_has * ((torch.cumsum(row_has, dim=2) - row_has) < 0.5).float()
        bottom_row = row_has * ((torch.flip(torch.cumsum(torch.flip(row_has, dims=[2]), dim=2), dims=[2]) - row_has) < 0.5).float()
        left_col = col_has * ((torch.cumsum(col_has, dim=3) - col_has) < 0.5).float()
        right_col = col_has * ((torch.flip(torch.cumsum(torch.flip(col_has, dims=[3]), dim=3), dims=[3]) - col_has) < 0.5).float()

        z_top = zero * top_row
        z_bottom = zero * bottom_row
        z_left = zero * left_col
        z_right = zero * right_col

        def leftmost(zr):
            return zr * ((torch.cumsum(zr, dim=3) - zr) < 0.5).float()
        def rightmost(zr):
            return zr * ((torch.flip(torch.cumsum(torch.flip(zr, dims=[3]), dim=3), dims=[3]) - zr) < 0.5).float()
        def topmost(zc):
            return zc * ((torch.cumsum(zc, dim=2) - zc) < 0.5).float()
        def bottommost(zc):
            return zc * ((torch.flip(torch.cumsum(torch.flip(zc, dims=[2]), dim=2), dims=[2]) - zc) < 0.5).float()

        tl = leftmost(z_top)
        tr = rightmost(z_top)
        bl = leftmost(z_bottom)
        br = rightmost(z_bottom)

        # A middle edge cell appears when leftmost == rightmost on a row/column.
        tm = tl * tr
        bm = bl * br
        lm = topmost(z_left) * bottommost(z_left)
        rm = topmost(z_right) * bottommost(z_right)
        return tl, tr, bl, br, tm, bm, lm, rm, row_has, col_has

    def forward(self, x):
        active = (x.sum(dim=1, keepdim=True) > 0.5).float()
        counts = x.sum(dim=(2, 3), keepdim=True)
        max_count = counts.amax(dim=1, keepdim=True)
        bg_color = (torch.abs(counts - max_count) < 0.5).float()

        # Seed color is the present nonzero non-background color.
        marker_color = ((counts > 0.5).float() * (1.0 - bg_color) * self.nonzero_color).clamp(0, 1)
        marker = (x * marker_color).sum(dim=1, keepdim=True)
        bg = (x * bg_color).sum(dim=1, keepdim=True) * active
        zero = x[:, 0:1] * active

        tl, tr, bl, br, tm, bm, lm, rm, row_has, col_has = self.endpoint_masks(zero)
        row_span = row_has.sum(dim=(2, 3), keepdim=True)
        col_span = col_has.sum(dim=(2, 3), keepdim=True)
        size3 = ((row_span > 2.5) & (row_span < 3.5) & (col_span > 2.5) & (col_span < 3.5)).float()
        size4 = ((row_span > 3.5) & (row_span < 4.5) & (col_span > 3.5) & (col_span < 4.5)).float()
        size5 = ((row_span > 4.5) & (row_span < 5.5) & (col_span > 4.5) & (col_span < 5.5)).float()

        zero_count = zero.sum(dim=(2, 3), keepdim=True)
        plus_side = ((zero_count > 4.5) & (zero_count < 5.5)).float()
        sparse_side = 1.0 - plus_side

        copied = torch.zeros_like(zero)

        # Corner regime: q is the trailing/opposite zero-template corner; s is bbox size.
        for anchor, sr, sc in [(tl, 1, 1), (tr, 1, -1), (bl, -1, 1), (br, -1, -1)]:
            for s, size_gate in [(3, size3), (4, size4), (5, size5)]:
                vr, vc = sr * s, sc * s
                gate = (anchor * self.shift(marker, -vr, -vc)).sum(dim=(2, 3), keepdim=True).clamp(0, 1) * size_gate
                ray = torch.zeros_like(zero)
                for k in range(1, 31):
                    dr, dc = k * vr, k * vc
                    if abs(dr) >= 30 or abs(dc) >= 30:
                        break
                    ray = torch.clamp(ray + self.shift(zero, dr, dc), 0, 1)
                copied = torch.clamp(copied + gate * ray, 0, 1)

        # Side regime: q is top/bottom-middle. Initial offset is 2*d; step is template-family dependent.
        for qmask, sr, sc in [(tm, 1, 1), (tm, 1, -1), (bm, -1, 1), (bm, -1, -1)]:
            ir, ic = 2 * sr, 2 * sc
            gate = (qmask * self.shift(marker, -ir, -ic)).sum(dim=(2, 3), keepdim=True).clamp(0, 1)

            ray_step1 = torch.zeros_like(zero)
            for n in range(0, 31):
                dr, dc = ir + n * sr, ic + n * sc
                if abs(dr) >= 30 or abs(dc) >= 30:
                    break
                ray_step1 = torch.clamp(ray_step1 + self.shift(zero, dr, dc), 0, 1)

            ray_step2 = torch.zeros_like(zero)
            for n in range(0, 31):
                dr, dc = ir + n * 2 * sr, ic + n * 2 * sc
                if abs(dr) >= 30 or abs(dc) >= 30:
                    break
                ray_step2 = torch.clamp(ray_step2 + self.shift(zero, dr, dc), 0, 1)

            copied = torch.clamp(copied + gate * (sparse_side * ray_step1 + plus_side * ray_step2), 0, 1)

        # Preserve all original pixels; only background cells receive the seed color.
        fill = copied * bg
        out = x * (1.0 - fill) + marker_color * fill
        return out * active

model = Task370QStepRelationBank().eval()
print('Model created')


Model created


In [8]:

# Direct PyTorch validation before export.
def validate_torch(split_name, examples):
    ok = 0
    bad = []
    outside_ok = 0
    with torch.no_grad():
        for i, ex in enumerate(examples):
            x_np = grid_to_tensor(ex['input'])
            exp = grid_to_tensor(ex['output'])
            y = model(torch.from_numpy(x_np)).numpy()
            pred = (y > 0.5).astype(np.float32)
            exact = np.array_equal(pred, exp)
            ok += int(exact)
            if not exact and len(bad) < 8:
                bad.append(i)
            active = active_canvas_from_tensor(x_np)
            outside_ok += int(np.all(pred * (1 - active) == 0))
    print(f'{split_name}: {ok}/{len(examples)} exact, outside_zero={outside_ok}/{len(examples)}, bad={bad}')
    return ok, bad

train_ok, train_bad = validate_torch('train', task['train'])
test_ok, test_bad = validate_torch('test', task['test'])
arc_ok, arc_bad = validate_torch('arc-gen', task.get('arc-gen', []))
assert train_ok == len(task['train'])
assert test_ok == len(task['test'])
assert arc_ok == len(task.get('arc-gen', []))


train: 3/3 exact, outside_zero=3/3, bad=[]
test: 1/1 exact, outside_zero=1/1, bad=[]
arc-gen: 262/262 exact, outside_zero=262/262, bad=[]


In [9]:

# Export ONNX.
dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)
# Use a real example as dummy-like sample to keep tracing stable with all active branches represented.
dummy = torch.from_numpy(grid_to_tensor(task['train'][0]['input']))

with torch.no_grad():
    torch.onnx.export(
        model,
        dummy,
        ONNX_PATH.as_posix(),
        input_names=['input'],
        output_names=['output'],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes=None,
        dynamo=False,
    )
print('Exported:', ONNX_PATH, 'size:', ONNX_PATH.stat().st_size)


/tmp/ipykernel_16/2237708850.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported: /kaggle/working/task370_qstep_relation_bank_v6_onnx/task370.onnx size: 805661


In [10]:

# ONNX structural checks.
onnx_model = onnx.load(ONNX_PATH.as_posix())
onnx.checker.check_model(onnx_model)
for value_info in list(onnx_model.graph.input) + list(onnx_model.graph.output):
    dims = [d.dim_value for d in value_info.type.tensor_type.shape.dim]
    print(value_info.name, dims)

input_shape = [d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]
output_shape = [d.dim_value for d in onnx_model.graph.output[0].type.tensor_type.shape.dim]
assert input_shape == [1, 10, 30, 30], input_shape
assert output_shape == [1, 10, 30, 30], output_shape

forbidden = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
ops = Counter(node.op_type for node in onnx_model.graph.node)
used_forbidden = sorted(set(ops) & forbidden)
print('ops:', dict(ops))
print('forbidden:', used_forbidden)
assert not used_forbidden
assert ONNX_PATH.stat().st_size < 1_400_000, ONNX_PATH.stat().st_size


input [1, 10, 30, 30]
output [1, 10, 30, 30]
ops: {'Constant': 2181, 'ReduceSum': 25, 'Greater': 11, 'Cast': 15, 'ReduceMax': 1, 'Sub': 10, 'Abs': 1, 'Less': 14, 'Mul': 200, 'Clip': 281, 'Slice': 519, 'CumSum': 6, 'And': 10, 'Concat': 256, 'Add': 269}
forbidden: []


In [11]:

# ONNX Runtime tensor-exact validation.
sess = ort.InferenceSession(ONNX_PATH.as_posix(), providers=['CPUExecutionProvider'])
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

def validate_ort(split_name, examples):
    ok = 0
    outside_ok = 0
    active_exact_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        x_np = grid_to_tensor(ex['input'])
        expected = grid_to_tensor(ex['output'])
        y = sess.run([output_name], {input_name: x_np})[0]
        pred = (y > 0.5).astype(np.float32)
        active = active_canvas_from_tensor(x_np)
        exact = np.array_equal(pred, expected)
        ok += int(exact)
        outside_ok += int(np.all(pred * (1 - active) == 0))
        active_exact_ok += int(np.array_equal(pred * active, expected * active))
        if not exact and len(bad) < 8:
            bad.append(i)
    print(f'{split_name}: exact={ok}/{len(examples)}, active_exact={active_exact_ok}/{len(examples)}, outside_zero={outside_ok}/{len(examples)}, bad={bad}')
    return ok, active_exact_ok, outside_ok, bad

summary = {}
for split_name in ['train', 'test', 'arc-gen']:
    examples = task.get(split_name, [])
    ok, active_ok, outside_ok, bad = validate_ort(split_name, examples)
    summary[split_name] = {
        'exact': ok,
        'total': len(examples),
        'active_exact': active_ok,
        'outside_zero': outside_ok,
        'bad': bad,
    }

assert summary['train']['exact'] == len(task['train'])
assert summary['test']['exact'] == len(task['test'])
assert summary['arc-gen']['exact'] == len(task.get('arc-gen', []))

summary['onnx_size_bytes'] = ONNX_PATH.stat().st_size
summary['input_shape'] = input_shape
summary['output_shape'] = output_shape
summary['forbidden_ops'] = used_forbidden
summary['ops'] = dict(ops)
with open(OUT_DIR / 'task370_validation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
summary


train: exact=3/3, active_exact=3/3, outside_zero=3/3, bad=[]
test: exact=1/1, active_exact=1/1, outside_zero=1/1, bad=[]
arc-gen: exact=262/262, active_exact=262/262, outside_zero=262/262, bad=[]


{'train': {'exact': 3,
  'total': 3,
  'active_exact': 3,
  'outside_zero': 3,
  'bad': []},
 'test': {'exact': 1,
  'total': 1,
  'active_exact': 1,
  'outside_zero': 1,
  'bad': []},
 'arc-gen': {'exact': 262,
  'total': 262,
  'active_exact': 262,
  'outside_zero': 262,
  'bad': []},
 'onnx_size_bytes': 805661,
 'input_shape': [1, 10, 30, 30],
 'output_shape': [1, 10, 30, 30],
 'forbidden_ops': [],
 'ops': {'Constant': 2181,
  'ReduceSum': 25,
  'Greater': 11,
  'Cast': 15,
  'ReduceMax': 1,
  'Sub': 10,
  'Abs': 1,
  'Less': 14,
  'Mul': 200,
  'Clip': 281,
  'Slice': 519,
  'CumSum': 6,
  'And': 10,
  'Concat': 256,
  'Add': 269}}

In [12]:

# Package submission and reproducibility bundle.
for zip_path in [KAGGLE_SUBMISSION_ZIP]:
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
    print('Wrote', zip_path, 'size', zip_path.stat().st_size)

# Bundle is created after this notebook is saved/executed by the driver script.
print('ONNX ready:', ONNX_PATH)


Wrote /kaggle/working/submission.zip size 46755
ONNX ready: /kaggle/working/task370_qstep_relation_bank_v6_onnx/task370.onnx
